<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
"""
One row = one content item, for one client, for one calendar month.
this notebook's window is March 2026 (month=2026-03), restricted to clients whose gsc_data_start/ga4_data_start falls on or before that month.
"""

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
"""
FEATURE (scored / gate signals — carry real temporal risk, get the leakage check in Part 3):
- gsc_impressions, gsc_clicks -> build ctr
- gsc_avg_position
- ga4_engaged_sessions, ga4_sessions -> build engagement_rate
- scroll_events, ga4_sessions -> build scroll_rate
- trend_pct (computed: current-month vs prior-month) and days_since_last_update
  (computed from content_updated_date) -> the gate

LABEL / PROXY (not a stored field - the formula that combines the features above):
- Refresh-priority score: gated by trend x staleness, scored by CTR/position/
  engagement/scroll, normalized by content_type/word_count, modified by
  search_volume/competition

CONTEXT (needed for the pipeline, never scored directly, no leakage check required):
- client_hash_id, content_hash_id -> join keys
- content_type, word_count -> normalizers
- search_volume, competition -> modifiers
- gsc_data_start, ga4_data_start, has_gsc_access, has_ga4_access (dim_clients)
  -> window validation only
- ga4_data_available, gsc_data_available -> availability filters

EXCLUDED (with reasons):
- provider_used, model_used -> production metadata, not a performance signal;
  risks the model learning "which AI wrote this" instead of "does it need refresh"
- is_published, is_deleted, is_active -> filter criteria applied before scoring,
  not features
- trend_direction / is_declining_label-equivalents -> downstream restatements of
  trend_pct; circular if used as features
- fact_content_query_90d (whole table) -> query-diversity signals are real but
  deliberately excluded to keep the feature set small, explainable, and less
  prone to overfitting
"""

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

In [3]:
daily_march_check = con.sql(f"""
    SELECT
        COUNT(*)              AS row_count,
        MIN(report_date)      AS min_date,
        MAX(report_date)      AS max_date,
        COUNT(DISTINCT content_hash_id) AS distinct_content,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
daily_march_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date,distinct_content,distinct_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [4]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,gsc_available_rows,both_available_rows
0,9841378,413966.0,3611061.0,364347.0


In [7]:
monthly_agg = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks)      AS gsc_clicks
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id, DATE_TRUNC('month', report_date)
""").df()
monthly_agg

,content_hash_id,client_hash_id,month,gsc_impressions,gsc_clicks
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,57.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,6523.0,7.0
2,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,453.0,0.0
3,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,2770.0,16.0
4,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,48.0,0.0
...,...,...,...,...,...
331432,content_ce93c3d40dc0dacb,client_3f0ce4d44fe94f3d,2026-03-01,0.0,0.0
331433,content_58793207f062aea6,client_3f0ce4d44fe94f3d,2026-03-01,0.0,0.0
331434,content_0fc0e9d2a98f4b47,client_3f0ce4d44fe94f3d,2026-03-01,0.0,0.0
331435,content_2713aadb9f2b4bb4,client_86ebc2f12c01f586,2026-03-01,0.0,0.0


In [8]:
grain_check = con.sql("""
    SELECT content_hash_id, client_hash_id, month, COUNT(*) AS n
    FROM monthly_agg
    GROUP BY content_hash_id, client_hash_id, month
    HAVING COUNT(*) > 1
""").df()

print(f"Aggregated rows: {len(monthly_agg)} | Duplicate groups (should be 0): {len(grain_check)}")

Aggregated rows: 331437 | Duplicate groups (should be 0): 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.